# Phase 10: Model Training — TensorFlow Deep Neural Network

## 🎯 Objective
Train a multi-output feed-forward **TensorFlow DNN** to capture global continuous non-linear representations
of pollutant dynamics across all 72 prediction horizons.

### Architecture
```
Input(64) → Dense(128, ReLU) + Dropout(0.3)
          → Dense(64, ReLU) + Dropout(0.2)
          → Dense(32, ReLU)
          → Dense(72, Linear)
```

### Training Protocol
- **Optimizer**: Adam (lr=0.001, with ReduceLROnPlateau)
- **Loss**: Mean Squared Error (aligned with RMSE-primary evaluation)
- **EarlyStopping**: patience=10, restore_best_weights=True
- **Validation**: Last 15% of training data (chronological tail) — test partition NEVER seen during training
- **Evaluation**: Same `ModelEvaluator` protocol as Phases 8–9, on identical `(X_test, y_test)`

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT.resolve()))

MODELS_DIR = PROJECT_ROOT / "data" / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

## 1. Official 4-Way Model Benchmark Comparison (Test Partition)

In [ ]:
with open(MODELS_DIR / "model_comparison.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

from src.training_pipeline.evaluator import ModelEvaluator
df_comp = ModelEvaluator.compare_models(eval_data)
display(df_comp)

## 2. Multi-Horizon Error Curves — 4-Way Comparison

In [ ]:
horizons = np.arange(1, 73)
models = {
    'Naive Persistence Baseline': {'color': 'gray', 'ls': '--', 'lw': 2},
    'Ridge Regression (MultiOutput)': {'color': '#4A90D9', 'ls': '-', 'lw': 2},
    'Random Forest Regressor': {'color': '#00E400', 'ls': '-', 'lw': 2},
    'TensorFlow DNN': {'color': '#FF6B35', 'ls': '-', 'lw': 2.5},
}

plt.figure(figsize=(12, 5))
for m_name, style in models.items():
    if m_name in eval_data:
        rmse_vals = [eval_data[m_name]['all_horizons'][f'h{h}']['rmse'] for h in horizons]
        plt.plot(horizons, rmse_vals, label=m_name, **style)

plt.axvline(24, color='orange', linestyle=':', alpha=0.7, label='24h (Day 1)')
plt.axvline(48, color='purple', linestyle=':', alpha=0.7, label='48h (Day 2)')
plt.axvline(72, color='red', linestyle=':', alpha=0.7, label='72h (Day 3)')

plt.title('4-Way Multi-Horizon Error Comparison (RMSE vs Forecast Step h)')
plt.xlabel('Forecast Horizon (Hours ahead)')
plt.ylabel('Root Mean Squared Error (EPA AQI points)')
plt.grid(alpha=0.3)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. TensorFlow Training History — Loss and Learning Rate Curves

In [ ]:
with open(MODELS_DIR / 'tf_training_history.json', 'r', encoding='utf-8') as f:
    tf_hist = json.load(f)

print(f"Epochs completed: {tf_hist['epochs_completed']}")
print(f"Best epoch:       {tf_hist['best_epoch']}")
print(f"Best val loss:    {tf_hist['best_val_loss']:.2f}")
print(f"Final train loss: {tf_hist['final_train_loss']:.2f}")
print(f"Final val loss:   {tf_hist['final_val_loss']:.2f}")

## 4. Short / Medium / Long Horizon Breakdown

In [ ]:
horizon_groups = {
    'Short (h1-h6)': list(range(1, 7)),
    'Medium (h7-h24)': list(range(7, 25)),
    'Long (h25-h72)': list(range(25, 73)),
}

records = []
for m_name in eval_data:
    for group_name, h_range in horizon_groups.items():
        rmse_vals = [eval_data[m_name]['all_horizons'][f'h{h}']['rmse'] for h in h_range]
        mae_vals = [eval_data[m_name]['all_horizons'][f'h{h}']['mae'] for h in h_range]
        records.append({
            'Model': m_name,
            'Horizon Group': group_name,
            'Avg RMSE': round(np.mean(rmse_vals), 2),
            'Avg MAE': round(np.mean(mae_vals), 2),
        })

df_groups = pd.DataFrame(records)
display(df_groups.pivot(index='Model', columns='Horizon Group', values='Avg RMSE'))

## 5. Key Findings & Progression to Phase 11

### Results Summary
- **TensorFlow DNN** achieves Overall RMSE = **85.01** (−1.15% vs Naive Baseline, behind Ridge's +1.29%).
- **Short horizons**: TF DNN RMSE at h+1 = **55.59** (17.4% better than Naive 67.30, competitive with RF 54.28 and Ridge 53.18).
- **Long horizons**: All learned models degrade beyond h+37, with the persistence baseline maintaining its local-level advantage.

### Validation Protocol Consistency
- All 4 models evaluated on the **same identical test partition** (`X_test`: 9,748 samples).
- TF early stopping used **training-only chronological validation** (last 15% of `X_train`).
- Walk-forward cross-validation is deferred to **Phase 11** for all models simultaneously.

### Phase 11: Walk-Forward Validation & Final Model Selection
Phase 11 will apply walk-forward expanding-window folds consistently across all 4 models to:
1. Verify seasonal stability (winter smog vs. summer monsoon).
2. Produce confidence intervals on RMSE estimates.
3. Select the final production model based on robust multi-season RMSE.